In [ ]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
X = torch.tensor([[1,2],[3,4]], dtype = torch.float32)
# Tensors are arrays that keep track of computational graph for autograd, every operation on a tensor builds a graph node, for backprop to walk back

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x**2
y.backward()
print(x.grad)
# requires_grad=True → track this tensor in the graph
# .backward() → compute all gradients w.r.t. graph leaves
# Only works for scalar outputs. For vectors, you need to provide gradient argument

In [ ]:
x = torch.tensor([2.0,1.0], requires_grad=True)
y = x**2
gradient_weights = torch.tensor([1.0,2.0])
y.backward(gradient = gradient_weights)
print(x.grad)
# Normal gradients   -  2x => [4,2]
# Weighted gradients -  2*x @ elementwise gradient_weights - [4,2] . [1,2] => [4,4]

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
class SimpleNet(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim,hidden_dim) 
         # fc = full connected layer / linear layer -> applies one linear transformation, weights nd biases initialisation range is +-1/root(n_samples), general purpose not He / Xavier at first
        self.fc2 = nn.Linear(hidden_dim,1)

    def forward(self,x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)
    
    def fit(self,x,y,epochs=1000,batch=32):

        dataset = TensorDataset(x, y)
        loader = DataLoader(dataset, batch_size=batch, shuffle=True)
        
        optimizer = torch.optim.SGD(self.parameters(), lr=0.00001, momentum=0.9)
        self.train()
        for epoch in range(epochs):
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                loss = torch.nn.functional.mse_loss(self(batch_x), batch_y)
                loss.backward()
                optimizer.step()

    def fit_gd(self,x,y,epochs=1000):
            
            optimizer = torch.optim.SGD(self.parameters(), lr=0.00001, momentum=0.9)
            self.train()
            for epoch in range(epochs):
                optimizer.zero_grad()
                y_pred = self(x) 
                loss = torch.nn.functional.mse_loss(y_pred, y)
                loss.backward()
                optimizer.step()
                
    def predict(self,x):
        self.eval()
        with torch.no_grad():
            return self(x)


In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_raw, y_raw = make_regression(n_samples=200, n_features=1, noise=10.0, random_state=42)

# Standardize features - 0 mean and unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_scaled, y_raw, test_size=0.25, random_state=42)

X_train, y_train = torch.FloatTensor(X_train_np), torch.FloatTensor(y_train_np).view(-1, 1)
X_test, y_test = torch.FloatTensor(X_test_np), torch.FloatTensor(y_test_np).view(-1, 1)

model_SGD = SimpleNet(1, 10)
model_GD = SimpleNet(1, 10)

model_SGD.fit(X_train, y_train, epochs=1000, batch=32)
model_GD.fit_gd(X_train, y_train, epochs=1000)

with torch.no_grad():
    print(f"Mini-batch Loss:    {torch.nn.functional.mse_loss(model_SGD.predict(X_test), y_test).item():.4f}")
    print(f"Full-Batch GD Loss: {torch.nn.functional.mse_loss(model_GD.predict(X_test), y_test).item():.4f}")

In [ ]:
epochs = 100
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
class DeepNet(nn.Module):
    def __init__(self,input_dim,hidden_dim=[20,20]):
        super().__init__()
        self.fc1 = nn.Linear(input_dim,hidden_dim[0]) 
        self.fc2 = nn.Linear(hidden_dim[0],hidden_dim[1])
        self.fc3 = nn.Linear(hidden_dim[1],1)
        self.history = {"sgd": [], "gd":[]}
        self.loss_history = {"sgd": [], "gd": []}
    def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)
    
    def fit(self,x,y,epochs=1000,batch=32):

        dataset = TensorDataset(x, y)
        loader = DataLoader(dataset, batch_size=batch, shuffle=True)
        
        optimizer = torch.optim.SGD(self.parameters(), lr=1e-6, momentum=0.9)
        self.train()
        for epoch in range(epochs):
            epoch_loss=0
            batch_norms = []
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                loss = torch.nn.functional.mse_loss(self(batch_x), batch_y)
                loss.backward()
                norms = {name: torch.norm(p.grad).item() for name, p in self.named_parameters() if p.grad is not None}
                batch_norms.append(norms)
                optimizer.step()
                epoch_loss += loss.item()
            avg_epoch_norm = {k: sum(b[k] for b in batch_norms)/len(batch_norms) for k in batch_norms[0].keys()}
            self.history["sgd"].append(avg_epoch_norm)
            self.loss_history["sgd"].append(epoch_loss / len(loader))
        
    def fit_gd(self,x,y,epochs=1000):
            
            optimizer = torch.optim.SGD(self.parameters(), lr=1e-5, momentum=0.9)
            self.train()
            for epoch in range(epochs):
                optimizer.zero_grad()
                y_pred = self(x) 
                loss = torch.nn.functional.mse_loss(y_pred, y)
                loss.backward()
                norms = {name: torch.norm(p.grad).item() for name, p in self.named_parameters() if p.grad is not None}
                self.history["gd"].append(norms)
                self.loss_history["gd"].append(loss.item())
                optimizer.step()
            
    def predict(self,x):
        self.eval()
        with torch.no_grad():
            return self(x)


In [ ]:
import matplotlib.pyplot as plt
model_SGD.fit(X_train, y_train, epochs=500, batch=32)
model_GD.fit_gd(X_train, y_train, epochs=500)


def plot_history(model, title):
    hist = model.history[title.lower()[:3]]
    if not hist: return
    
    plt.figure(figsize=(10, 4))

    for key in [k for k in hist[0].keys() if 'weight' in k]:
        plt.plot([e[key] for e in hist], label=key)
    
    plt.yscale('log')
    plt.title(f"Gradient Norms: {title}")
    plt.xlabel("Epoch")
    plt.ylabel("Norm (Log)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_history(model_SGD, "SGD")
plot_history(model_GD, "GD")
print("GD WEIGHTS")
print("FINAL GD GRADIENT NORMS")
hist_gd = model_GD.history["gd"]
for key in [k for k in hist_gd[0].keys() if 'weight' in k]:
    print(f"{key}: {hist_gd[-1][key]:.6f}")

print("\nFINAL SGD GRADIENT NORMS")
hist_sgd = model_SGD.history["sgd"]
for key in [k for k in hist_sgd[0].keys() if 'weight' in k]:
    print(f"{key}: {hist_sgd[-1][key]:.6f}")
with torch.no_grad():
    print(f"\nFinal SGD Test Loss: {torch.nn.functional.mse_loss(model_SGD.predict(X_test), y_test).item():.4f}")
    print(f"Final GD Test Loss:  {torch.nn.functional.mse_loss(model_GD.predict(X_test), y_test).item():.4f}")

In [ ]:
def analyze_training(model):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(model.loss_history["sgd"], label="SGD Loss", color='red', alpha=0.7)
    ax1.plot(model.loss_history["gd"], label="GD Loss", color='blue', alpha=0.7)
    ax1.set_yscale('log')
    ax1.set_title("Training Loss (Convergence)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("MSE Loss (Log)")
    ax1.legend()
    ax1.grid(True)


    hist_sgd = model.history["sgd"]
    for key in [k for k in hist_sgd[0].keys() if 'weight' in k]:
        ax2.plot([e[key] for e in hist_sgd], label=key)
    
    ax2.set_yscale('log')
    ax2.set_title("SGD Gradient Norms (Stability)")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Norm (Log)")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

analyze_training(model_SGD)

In [ ]:
from torchvision import datasets, transforms
class MLPNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Flattened input is 28*28 = 784
        self.fc1 = nn.Linear(28*28, 128)
        # self.fc2 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        # self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        # x = self.dropout(x) 
        x = torch.relu(self.fc2(x))
        # x = self.dropout(x)
        # x = torch.relu(self.fc3(x))
        return self.fc3(x)

    def fit(self, loader, epochs=5):
        criterion = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(self.parameters(), lr=0.1, momentum=0.9)
        # optimizer = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=1e-4)
        
        self.train()
        last_batch_loss = 0
        for epoch in range(epochs):
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                loss = criterion(self(batch_x), batch_y)
                loss.backward()
                optimizer.step()
                last_batch_loss = loss.item()
        return last_batch_loss

    def evaluate(self, loader):
        self.eval()
        criterion = nn.CrossEntropyLoss()
        total_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in loader:
                loss = criterion(self(batch_x), batch_y)
                total_loss += loss.item()
        
        return total_loss / len(loader)

In [ ]:
g = torch.Generator()
g.manual_seed(42)

transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data,batch_size=64,shuffle=True,generator=g)

test_loader = torch.utils.data.DataLoader(test_data,batch_size=64,shuffle=False)

model = MLPNet()
train_loss = model.fit(train_loader)
print(f"Final Training Loss: {train_loss}")

test_loss = model.evaluate(test_loader)
print(f"Average Testing Loss:{test_loss}")

In [ ]:
# Adam doesnt always necessarily generalize worse than SGD / Adaptive methods change geometry
# Noise acts as implicit regularizer
# Norm growth is correlated to margin (over here not in general)
# Hyperparameters dominate the results

# Both generalize similarly with 0.01 SGD and 0.001 Adam, but 0.01 Adam shows scale drift , first weight explodes , norm increases so does margin and worse accuracy because its without any growth constraints

# SGD already has noise and Dropouts gradient variance addition is negligible
# Dropout increases variance -> adam scaling gets controlled as effective step size reduces and it doesnt explode and gives the best result

# Adam with 0.01 nd weight decay, high LR still dominates 
# Adam is more sensitive to LR.
# Dropout stabilizes adaptive scaling.
# Weight decay controls norm growth.
# SGD naturally keeps norms moderates
# Also this is with 5 epochs only, not asymptotic theory